# Pipeline CAGED SC — orquestração ponta a ponta (R8.1, R8.4)

Este notebook encadeia o pipeline do Datathon CDIA 2026 — *Raio-X do
Mercado de Trabalho e Vagas Industriais em Santa Catarina* — **sem
intervenção manual entre etapas** (R8.1), executando em sequência:

1. **Ingestão** (`src/ingestao.py`) → `data_processed/caged_bruto.parquet`
2. **Limpeza** (`src/limpeza.py` + `src/mapeamentos.py`)
3. **Junção relacional** (`src/enriquecimento.py`)
4. **Métricas de negócio** (`src/metricas.py`)
5. **Classificação de CBO via LLM** (`src/cbo_ia.py`)
6. **Agregação** (`src/agregador.py`) → `data_processed/caged_agregado.parquet`
7. **Upload para o Azure Blob Storage** (`src/azure_upload.py`)

Cada etapa registra, via `src/logging_pipeline.log_etapa`, **uma única
mensagem de progresso** identificando a etapa (R8.4). O artefato final
`caged_agregado.parquet` é o que o dashboard (`dashboard/app.py`) consome.

## Pré-requisitos e credenciais (variáveis de ambiente)

As credenciais são lidas **exclusivamente de variáveis de ambiente** e
nunca ficam versionadas no repositório (R8.5). Defina-as no shell antes
de abrir o notebook:

| Variável | Uso | Etapa guardada |
|---|---|---|
| `LLM_API_KEY` | credencial da API do LLM externo (Claude / Anthropic) | CBO/LLM — interrompe antes da API se ausente (R8.6) |
| `AZURE_STORAGE_CONNECTION_STRING` | connection string do Azure Storage | upload — interrompe sem enviar dados se ausente/inválida (R8.7, R9.3) |

A chave do LLM é obtida no **Anthropic Console** (console.anthropic.com) e
começa com `sk-ant-`.

```bash
export LLM_API_KEY="sk-ant-..."
export AZURE_STORAGE_CONNECTION_STRING="DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net"
```

O agrupamento de CBOs roda **localmente** via API externa do Claude
(Anthropic) — **não** usa Azure OpenAI (R5.9, R9.5). Somente as tabelas
agregadas finais sobem para o Blob_Storage.

## Configuração: imports, caminhos e logging

Adiciona a raiz do projeto ao `sys.path` para importar o pacote `src`,
define os caminhos de origem/saída e inicializa o logger do pipeline. As
mensagens `[etapa] ...` abaixo são emitidas por `log_etapa` (R8.4).

In [1]:
import os
import sys
from pathlib import Path

# Raiz do projeto = pasta que contém `src/`, `Dados/` e `data_processed/`.
# Em um notebook, `__file__` não existe; partimos do diretório de trabalho.
RAIZ = Path.cwd()
if not (RAIZ / "src").is_dir() and (RAIZ.parent / "src").is_dir():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.logging_pipeline import obter_logger, log_etapa

logger = obter_logger()

# --- Caminhos de origem (R1.1, R1.2) ---
# Fonte do CAGED (M5.C5): o consolidado de 24 meses (MOV + FOR − EXC do FTP do
# MTE, `python -m src.download && python -m src.caged24`, validado contra a base
# fornecida: +0,02%) quando existe; `FONTE_CAGED=fornecida` força a base de 12 meses.
CAMINHO_CAGED_24M = RAIZ / "data" / "interim" / "caged_sc_24m.parquet"
USAR_24M = CAMINHO_CAGED_24M.exists() and os.environ.get("FONTE_CAGED") != "fornecida"
CAMINHO_CAGED = str(CAMINHO_CAGED_24M if USAR_24M else RAIZ / "Dados" / "Novo_Caged_MOV_Limpo.csv")
# Janela: set/24–ago/26 com 24 meses; set/25–ago/26 com a base fornecida (D1).
COMPETENCIA_INICIO = 202409 if USAR_24M else 202509
CAMINHO_MUNICIPIOS = str(RAIZ / "Dados" / "Municipios_SC_limpo.csv")

# --- Caminhos de saída (artefatos consumidos pelo dashboard) ---
DIR_PROCESSADO = RAIZ / "data_processed"
DIR_PROCESSADO.mkdir(parents=True, exist_ok=True)
CAMINHO_PARQUET_BRUTO = str(DIR_PROCESSADO / "caged_bruto.parquet")
CAMINHO_CSV_CBO = str(DIR_PROCESSADO / "cbo_macrofamilias.csv")
CAMINHO_AGREGADO = str(DIR_PROCESSADO / "caged_agregado.parquet")
CAMINHO_PERFIL = str(DIR_PROCESSADO / "caged_perfil.parquet")
CAMINHO_METADADOS = str(DIR_PROCESSADO / "metadados.json")

# Container de destino no Blob_Storage para as tabelas agregadas (R9.1).
CONTAINER_BLOB = os.environ.get("AZURE_BLOB_CONTAINER", "caged-agregado")

print(f"Raiz do projeto: {RAIZ}")
print(f"CAGED:       {CAMINHO_CAGED}")
print(f"Municípios:  {CAMINHO_MUNICIPIOS}")
print(f"Agregado:    {CAMINHO_AGREGADO}")

Raiz do projeto: /workspaces/DataThon
CAGED:       /workspaces/DataThon/data/interim/caged_sc_24m.parquet
Municípios:  /workspaces/DataThon/Dados/Municipios_SC_limpo.csv
Agregado:    /workspaces/DataThon/data_processed/caged_agregado.parquet


## Etapa 1 — Ingestão (R1)

`executar_ingestao` lê o CAGED (sep `;`, decimal `,`) e a Base_de_Municipios
(sep `,`), descarta as colunas de índice `Unnamed:*`, valida `uf == 42` e a
presença das colunas obrigatórias e, **somente após as validações**,
persiste `caged_bruto.parquet` (R1.3–R1.8). Qualquer arquivo ausente/ilegível
ou validação reprovada interrompe o pipeline antes de gravar (R1.10).

In [2]:
from src.ingestao import executar_ingestao, ler_municipios

resultado_ingestao = executar_ingestao(
    caminho_caged=CAMINHO_CAGED,
    caminho_municipios=CAMINHO_MUNICIPIOS,
    destino_parquet=CAMINHO_PARQUET_BRUTO,
)

print(resultado_ingestao)

2026-11-13 02:26:50,895 | INFO | datathon.pipeline | [ingestão] 6782144 linhas do CAGED e 296 de municípios lidas; Parquet bruto gravado em '/workspaces/DataThon/data_processed/caged_bruto.parquet'.


IngestResult(linhas_caged=6782144, linhas_municipios=296, destino_parquet='/workspaces/DataThon/data_processed/caged_bruto.parquet')


## Etapa 2 — Limpeza e decodificação de domínios (R2)

Carrega os domínios do `Dicionario_de_dados.xlsx` (`src/mapeamentos.py`),
relê o Parquet bruto e os municípios, e aplica `limpar`: conversão monetária
BR → float no CAGED, **conversão com ponto decimal** nos indicadores
municipais (o arquivo do IBGE não está em formato brasileiro), decodificação
de domínios e filtro de competência `[202509, 202608]`.

`limpar` também deriva, **antes** da decodificação e a partir do código cru,
a coluna `escolaridade_ordinal` — o nível ordinal de `graudeinstrução`, que é
o eixo contínuo da dispersão renda × escolaridade (R4.7, R7.4). Decodificada
para rótulo, a coluna original não seria ordenável.

In [3]:
import pandas as pd

from src.mapeamentos import (
    GRAU_INSTRUCAO,
    RACA_COR,
    SEXO,
    TIPO_MOVIMENTACAO,
    CATEGORIA,
    TAM_ESTAB_JAN,
    UNIDADE_SALARIO,
    SECAO_CNAE,
)
from src.limpeza import limpar

# Relê o Parquet bruto persistido pela ingestão e a base de municípios.
df_caged_bruto = pd.read_parquet(CAMINHO_PARQUET_BRUTO)
df_municipios_bruto = ler_municipios(CAMINHO_MUNICIPIOS)

# Mapas de domínio código → rótulo por coluna categórica (R2.5).
mapas_dominio = {
    "graudeinstrução": GRAU_INSTRUCAO,
    "raçacor": RACA_COR,
    "sexo": SEXO,
    "tipomovimentação": TIPO_MOVIMENTACAO,
    "categoria": CATEGORIA,
    "tamestabjan": TAM_ESTAB_JAN,
    "unidadesaláriocódigo": UNIDADE_SALARIO,
    "seção": SECAO_CNAE,
}

resultado_limpeza = limpar(
    df_caged_bruto, df_municipios_bruto, mapas_dominio, inicio=COMPETENCIA_INICIO
)

print(
    f"CAGED limpo: {len(resultado_limpeza.caged)} linhas; "
    f"excluídas por competência: {resultado_limpeza.excluidos_competencia}"
)
print(
    "escolaridade_ordinal numérica em "
    f"{resultado_limpeza.caged['escolaridade_ordinal'].notna().sum()} linhas."
)

# A base bruta só serve para a contagem de linhas daqui em diante: libera a memória
# (com 24 meses, cada cópia da base pesa ~1 GB).
LINHAS_BRUTAS = len(df_caged_bruto)
# BUG-02: linhas por tipo de arquivo (MOV, FOR, EXC) para o funil da Metodologia.
LINHAS_POR_TIPO = (df_caged_bruto["tipo_arquivo"].value_counts().to_dict()
                   if "tipo_arquivo" in df_caged_bruto.columns else None)
del df_caged_bruto


2026-11-13 02:27:28,027 | INFO | datathon.pipeline | [limpeza] 6782144 linhas mantidas no CAGED; 0 excluídas por competência; 0 grupo(s) de falha monetária; 0 código(s) categórico(s) não mapeado(s).


CAGED limpo: 6782144 linhas; excluídas por competência: 0
escolaridade_ordinal numérica em 6782144 linhas.


## Etapa 3 — Junção relacional CAGED × municípios (R3)

`juntar` normaliza as chaves municipais para 6 dígitos (CAGED de 6 dígitos;
IBGE de 7 → 6 primeiros), exclui chaves malformadas (contadas por base) e
executa um **left join** preservando todo o CAGED válido. Emite alerta se
mais de 1% dos registros ficar sem correspondência (R3.3–R3.6).

In [4]:
from src.enriquecimento import juntar

resultado_juncao = juntar(resultado_limpeza.caged, resultado_limpeza.municipios)
df_cruzado = resultado_juncao.df

print(
    f"Registros lidos: {resultado_juncao.total_caged}; "
    f"com chave válida: {resultado_juncao.total_caged_valido}; "
    f"sem correspondência: {resultado_juncao.sem_correspondencia}; "
    f"alerta de limiar: {resultado_juncao.alerta}"
)

2026-11-13 02:27:31,798 | INFO | datathon.pipeline | [junção] 6782144 registros lidos do CAGED (6782144 com chave válida); 0 sem município correspondente; excluídos por chave malformada: CAGED=0, municípios=1.


Registros lidos: 6782144; com chave válida: 6782144; sem correspondência: 0; alerta de limiar: False


## Etapa 4 — Métricas de negócio (R4)

As funções puras de `src/metricas.py` calculam, sobre a base cruzada, as
métricas que alimentam a agregação: salário médio por movimentação e
distribuição de escolaridade por seção. Saldo_Liquido, Saldo_por_10k,
Turnover e Indice_de_Aquecimento são recomputados por recorte dentro do
Agregador (etapa 6), que reaproveita essas mesmas funções puras.

In [5]:
from src.metricas import (
    salario_medio_por_movimentacao,
    distribuicao_escolaridade,
)

# A base enriquecida com a macrofamília (etapa 5) é a entrada da agregação;
# aqui calculamos as métricas por seção que compõem a leitura analítica.
salario_medio = salario_medio_por_movimentacao(df_cruzado)
escolaridade = distribuicao_escolaridade(df_cruzado)

log_etapa(
    "métricas",
    f"{len(salario_medio)} linha(s) de salário médio; "
    f"{len(escolaridade)} linha(s) de distribuição de escolaridade.",
)

print(salario_medio.head())

2026-11-13 02:27:31,823 | INFO | datathon.pipeline | [métricas] 0 linha(s) de salário médio; 0 linha(s) de distribuição de escolaridade.


Empty DataFrame
Columns: [movimentação, seção, salario_medio]
Index: []


## Etapa 5 — Classificação de CBO via LLM externo (R5)

`classificar_cbos` extrai os CBOs distintos, particiona em lotes de ≤100,
chama o LLM externo (retry ≤3, timeout 60 s), consolida 1 macrofamília por
CBO (fallback `Não classificado`), persiste `cbo_macrofamilias.csv` + o
prompt usado, e faz o merge na base (R5.1–R5.8).

O provedor é escolhido por `LLM_PROVIDER`: **`gemini`** (padrão — Google AI
Studio, camada gratuita) ou `anthropic`. Os adaptadores vivem em
`src/cbo_ia.py` por trás da interface `ClienteLLM`, de modo que trocar de
provedor não toca no pipeline.

Dois comportamentos protegem contra o modo de falha silencioso:

- **Falha permanente** (saldo insuficiente, credencial inválida, modelo
  inexistente — HTTP 400/401/403/404) **interrompe** o pipeline, em vez de
  virar `Não classificado`. É problema de configuração, e repetir não resolve.
- Se mesmo assim mais da metade dos CBOs cair no fallback, a etapa registra um
  **ALERTA** no log.

In [6]:
from src.cbo_ia import (
    VAR_PROVEDOR_LLM,
    aplicar_mapeamento,
    classificar_cbos,
    criar_cliente_llm,
)
import os

# A classificação por IA é cara e não determinística; uma vez gravada em
# `cbo_macrofamilias.csv` (revisável e versionado), as próximas execuções a
# reaproveitam. `RECLASSIFICAR_CBO=1` força uma nova rodada no LLM.
if Path(CAMINHO_CSV_CBO).exists() and not os.environ.get("RECLASSIFICAR_CBO"):
    print(f"Classificação de CBO reaproveitada de {CAMINHO_CSV_CBO} (sem chamar o LLM).")
else:
    # `criar_cliente_llm` aplica a guarda de credencial (R8.6) — interrompe antes
    # de qualquer chamada à API se `LLM_API_KEY` estiver ausente/vazia — e escolhe
    # o provedor por `LLM_PROVIDER`: "gemini" (padrão), "anthropic" ou "arquivo"
    # (respostas da IA paga coladas em ia_cbo/respostas/, sem chave; README, Opção C).
    cliente_llm = criar_cliente_llm()
    print(f"Provedor do LLM_Externo: {os.environ.get(VAR_PROVEDOR_LLM, 'gemini')} "
          f"({type(cliente_llm).__name__})")
    resultado_cbo = classificar_cbos(df_cruzado, cliente_llm, destino_csv=CAMINHO_CSV_CBO)

# Depois da saída da IA: a regra explícita de desempate entre as rodadas (TAX-04 P1)
# e os overrides manuais revisados (TAX-03, `ia_cbo/overrides.csv`). O mapeamento
# final, com a origem de cada decisão, vai para `cbo_classificacao_final.csv`.
from src.cbo_ia import CAMINHO_OVERRIDES, NOME_CSV_FINAL, NOME_CSV_REFERENCIA, mapeamento_final

CAMINHO_CSV_REFERENCIA = str(DIR_PROCESSADO / NOME_CSV_REFERENCIA)
mapa_final = mapeamento_final(CAMINHO_CSV_CBO, caminho_referencia=CAMINHO_CSV_REFERENCIA,
                              caminho_overrides=CAMINHO_OVERRIDES)
mapa_final.to_csv(DIR_PROCESSADO / NOME_CSV_FINAL, index=False)
df_enriquecido = aplicar_mapeamento(df_cruzado, CAMINHO_CSV_CBO, caminho_referencia=CAMINHO_CSV_REFERENCIA,
                                    caminho_overrides=CAMINHO_OVERRIDES)
print(mapa_final["origem"].value_counts())

print(df_enriquecido["macrofamilia"].value_counts())


Classificação de CBO reaproveitada de /workspaces/DataThon/data_processed/cbo_macrofamilias.csv (sem chamar o LLM).
macrofamilia
Administrativo e comercial               1952434
Serviços de apoio e outros               1594611
Logística e transporte                    845088
Operação de linha e processo              743953
Construção civil e extração               496064
Metalmecânica e eletroeletrônica          261418
Têxtil, confecção, couro e calçados       260987
Engenharia, técnicos industriais e TI     214699
Alimentos e bebidas                       171687
Manutenção e reparação                    161274
Madeira, móveis, cerâmica e vidro          79059
Não classificado                             684
Name: count, dtype: int64


## Etapa 6 — Agregação e exportação (R6)

`agregar` produz uma linha por (região/município × seção × macrofamília ×
mês), reaproveitando as funções puras de métricas, anexa população 2026 /
IDHM 2010 / PIB 2023, acrescenta salário médio de admissão/desligamento e
escolaridade média dos admitidos (fonte da dispersão do dashboard, R7.4) e
arredonda as métricas a 2 casas. `exportar` grava
`caged_agregado.parquet` de forma atômica — sem deixar arquivo parcial em
caso de falha (R6.1–R6.7).

Antes de agregar, o orquestrador deriva `chave_municipal` na base de
municípios a partir do código IBGE (6 primeiros dígitos), pois o anexo dos
indicadores socioeconômicos é feito por essa chave, e coage o saldo para
número (o CAGED é lido como texto na ingestão).

In [7]:
import json

from src.agregador import (
    agregar,
    agregar_cbo,
    agregar_perfil,
    escrever_metadados,
    exportar,
    montar_metadados,
    teto_salario,
)
from src.enriquecimento import chave_municipal_ibge, resolver_coluna_codigo

# Prep: `agregar` anexa os indicadores socioeconômicos por `chave_municipal`,
# mas a base de municípios limpa ainda não tem essa chave (ela só é derivada
# dentro de `juntar`, sobre o CAGED cruzado). Derivamos aqui a chave de 6
# dígitos a partir do código IBGE de 7 (`resolver_coluna_codigo` tolera o
# cabeçalho real `Código [-]` além do `Código` simples).
municipios_agg = resultado_limpeza.municipios.copy()
municipios_agg["chave_municipal"] = municipios_agg[
    resolver_coluna_codigo(municipios_agg)
].map(chave_municipal_ibge)

# O CAGED é lido como texto (dtype=str) na ingestão; o saldo precisa ser
# numérico antes de agregar. O Turnover sai das próprias contagens de
# movimentação, sem denominador externo.
base_metricas = df_enriquecido.copy(deep=False)  # rasa: copy-on-write
base_metricas["saldomovimentação"] = pd.to_numeric(
    base_metricas["saldomovimentação"], errors="coerce"
)

df_agregado = agregar(base_metricas, municipios_agg)
exportar(df_agregado, CAMINHO_AGREGADO)

# Perfil de causa raiz (medianas salariais por faixa de escolaridade) e a
# procedência dos artefatos, ambos lidos pelo dashboard.
df_perfil = agregar_perfil(base_metricas)
exportar(df_perfil, CAMINHO_PERFIL)

# Tabela por ocupação (CBO): o "o que formar" das recomendações e a razão
# salarial dentro da mesma CBO (Lote B do plano da banca).
CAMINHO_CBO_AGREGADO = str(DIR_PROCESSADO / "caged_cbo.parquet")
df_cbo = agregar_cbo(base_metricas)
exportar(df_cbo, CAMINHO_CBO_AGREGADO)

caminho_meta_ia = Path(CAMINHO_CSV_CBO + ".meta.json")
meta_ia = json.loads(caminho_meta_ia.read_text(encoding="utf-8")) if caminho_meta_ia.exists() else {}
metadados = montar_metadados(
    linhas_brutas=LINHAS_BRUTAS,
    linhas_por_tipo=LINHAS_POR_TIPO,
    linhas_limpas=len(resultado_limpeza.caged),
    excluidas_competencia=resultado_limpeza.excluidos_competencia,
    df_agregado=df_agregado,
    df_perfil=df_perfil,
    meta_ia=meta_ia,
    teto_salario_valido=teto_salario(base_metricas),
    raiz=RAIZ,
)
escrever_metadados(CAMINHO_METADADOS, metadados)

print(f"Agregado: {len(df_agregado)} linha(s); perfil: {len(df_perfil)} linha(s); metadados em {CAMINHO_METADADOS}")
df_agregado.head()

2026-11-13 02:28:22,242 | INFO | datathon.pipeline | [agregação] 512548 recortes agregados (município × seção × macrofamília × mês) em 6 mesorregião(ões).


2026-11-13 02:28:23,455 | INFO | datathon.pipeline | [agregação] agregado exportado para '/workspaces/DataThon/data_processed/caged_agregado.parquet' (512548 linhas).


2026-11-13 02:28:51,317 | INFO | datathon.pipeline | [agregação] perfil de causa raiz com 18975 linhas.


2026-11-13 02:28:51,401 | INFO | datathon.pipeline | [agregação] agregado exportado para '/workspaces/DataThon/data_processed/caged_perfil.parquet' (18975 linhas).


2026-11-13 02:29:08,119 | INFO | datathon.pipeline | [agregação] tabela por CBO com 464167 linhas.


2026-11-13 02:29:08,433 | INFO | datathon.pipeline | [agregação] agregado exportado para '/workspaces/DataThon/data_processed/caged_cbo.parquet' (464167 linhas).


2026-11-13 02:29:12,253 | INFO | datathon.pipeline | [agregação] metadados gravados em '/workspaces/DataThon/data_processed/metadados.json'.


Agregado: 512548 linha(s); perfil: 18975 linha(s); metadados em /workspaces/DataThon/data_processed/metadados.json


,regiao,municipio,secao_industrial,macrofamilia_cbo,mes,saldo_liquido,saldo_por_10k,salario_medio_admissao,salario_medio_desligamento,escolaridade_media,...,desligamentos_a_pedido,desligamentos_descontados,soma_salario_admissao,n_salario_admissao,soma_salario_desligamento,n_salario_desligamento,soma_escolaridade_admissao,n_escolaridade_admissao,soma_escolaridade_desligamento,n_escolaridade_desligamento
0,Serrana,420005,"Administração Pública, Defesa e Seguridade Social",Serviços de apoio e outros,202502,-1.0,-3.81,NaN,2788.49,NaN,...,1,0,0.00,0,2788.49,1,0.0,0,7.0,1
1,Serrana,420005,"Administração Pública, Defesa e Seguridade Social",Serviços de apoio e outros,202504,1.0,3.81,3175.25,NaN,9.5,...,0,0,3175.25,1,0.00,0,9.5,1,0.0,0
2,Serrana,420005,"Administração Pública, Defesa e Seguridade Social",Serviços de apoio e outros,202608,-1.0,-3.81,NaN,3242.00,NaN,...,1,0,0.00,0,3242.00,1,0.0,0,7.0,1
3,Serrana,420005,"Administração Pública, Defesa e Seguridade Social",Serviços de apoio e outros,202602,-1.0,-3.81,NaN,27738.88,NaN,...,0,1,0.00,0,27738.88,1,0.0,0,9.5,1
4,Serrana,420005,"Administração Pública, Defesa e Seguridade Social",Serviços de apoio e outros,202604,1.0,3.81,28429.48,NaN,9.0,...,0,0,28429.48,1,0.00,0,9.0,1,0.0,0


## Etapa 7 — Upload para o Azure Blob Storage (R9)

`upload_tabelas` lê `AZURE_STORAGE_CONNECTION_STRING` e, se ausente/inválida,
**interrompe antes de qualquer chamada ao Azure** — sem enviar dados parciais
(R8.7, R9.3). Caso contrário, envia a(s) tabela(s) agregada(s) ao container
informado (R9.1), de onde o dashboard as lê (com fallback local — R9.6).

In [8]:
import os

from src.azure_upload import upload_tabelas

# `PULAR_UPLOAD=1` gera os artefatos sem publicar (o upload é o passo do M7).
if os.environ.get("PULAR_UPLOAD"):
    enviados = []
    print("PULAR_UPLOAD definido: artefatos gerados em data_processed/, nada enviado ao Blob.")
else:
    # O estoque da RAIS (M4) sobe junto: o painel recusa schema misturado.
    estoque = DIR_PROCESSADO / "rais_estoque.parquet"
    enviados = upload_tabelas(
        [CAMINHO_AGREGADO, CAMINHO_PERFIL, CAMINHO_METADADOS, CAMINHO_CBO_AGREGADO]
        + ([str(estoque)] if estoque.exists() else []),
        CONTAINER_BLOB,
    )
    print(f"Tabelas enviadas ao container {CONTAINER_BLOB!r} do Blob_Storage: {enviados}")

PULAR_UPLOAD definido: artefatos gerados em data_processed/, nada enviado ao Blob.


## Conclusão e checagem de sanidade

O pipeline executou de ponta a ponta sem intervenção manual (R8.1),
registrando uma mensagem de progresso única por etapa (R8.4). A célula
abaixo afirma o que uma execução saudável precisa produzir — ela falha alto
caso uma das correções regrida.

In [9]:
# Checagem de sanidade da execução (falha alto se algo regredir).
idhm = df_agregado["idhm_2010"].dropna()
assert (idhm > 0).all() and (idhm <= 1).all(), f"IDHM fora de [0,1]: {idhm.min()}–{idhm.max()}"
pib = df_agregado["pib_per_capita_2023"].dropna()
assert pib.max() < 1_000_000, f"PIB per capita implausível: {pib.max():,.2f}"
assert df_agregado["salario_medio_admissao"].notna().any(), "salário médio ausente"
assert df_agregado["escolaridade_media"].notna().any(), "escolaridade média ausente"

familias = df_enriquecido["macrofamilia"].value_counts(normalize=True)
nao_classificado = familias.get("Não classificado", 0.0)
assert nao_classificado < 0.5, (
    f"{nao_classificado:.0%} dos vínculos sem macrofamília — "
    "o enriquecimento por IA não funcionou"
)

print("Checagem de sanidade OK.")
print(f"  recortes agregados ........ {len(df_agregado)}")
print(f"  IDHM ...................... {idhm.min():.3f} – {idhm.max():.3f}")
print(f"  PIB per capita (máx) ...... R$ {pib.max():,.2f}")
print(f"  macrofamílias distintas ... {df_enriquecido['macrofamilia'].nunique()}")
print(f"  'Não classificado' ........ {nao_classificado:.1%}")


Checagem de sanidade OK.
  recortes agregados ........ 512548
  IDHM ...................... 0.621 – 0.847
  PIB per capita (máx) ...... R$ 218,625.97


  macrofamílias distintas ... 12
  'Não classificado' ........ 0.0%


O artefato `data_processed/caged_agregado.parquet` está pronto para
consumo pelo dashboard (`dashboard/app.py`), com cópia no Blob_Storage para a
demonstração.

Para iniciar o dashboard:

```bash
streamlit run dashboard/app.py
```